# Fine-tune ViT5 trên Kaggle — tầng 3

Notebook này chạy `src/models/vit5.py` của repo DL-SummariseVN trên GPU T4 của Kaggle.

## Trước khi chạy: ba cài đặt trong panel **Settings** bên phải

| Mục | Đặt thành | Vì sao |
|---|---|---|
| **Accelerator** | `GPU T4 x2` | Kaggle không có tuỳ chọn một T4; ta nhận hai rồi tự ghim còn một ở ô dưới |
| **Internet** | `On` | Cần tải `VietAI/vit5-base` và bộ `nam194/vietnews` từ Hugging Face. Muốn bật thì tài khoản Kaggle phải xác minh số điện thoại |
| **Persistence** | `Files only` (tuỳ chọn) | Giữ `/kaggle/working` giữa các phiên tương tác |

## Ngân sách

- **30 giờ GPU mỗi tuần**, đặt lại vào thứ Bảy — nhiều hơn Colab free đáng kể.
- **Một phiên tối đa 12 giờ**. `train_5k` 3 epoch hết khoảng 60 phút, `train_10k`
  khoảng 2 giờ, `train_20k` khoảng 4 giờ — đều gọn trong một phiên.
- Phiên tương tác tự ngắt khi để yên quá lâu. Cách chạy ngầm nằm ở cuối notebook.

In [ ]:
!nvidia-smi

import datasets, torch, transformers
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("GPU thấy được:", torch.cuda.device_count())

## Vì sao phải ghim lại còn MỘT GPU

Kaggle cấp hai T4. `Trainer` thấy hai thiết bị sẽ tự bật DataParallel, và khi đó
`--batch 2` là **mỗi GPU** — batch hiệu dụng thành 32 chứ không phải 16 như README đã
chốt. Số liệu sinh ra sẽ không so được với lần chạy `train_5k` trên Colab, mà mục đích
của cả đề tài là **so sánh có kiểm soát**: đổi batch hiệu dụng giữa hai lần chạy là
đổi mất chính thứ đang được so.

Vì vậy mọi lệnh huấn luyện dưới đây đều mở đầu bằng `CUDA_VISIBLE_DEVICES=0`. Ghim
ngay trên dòng lệnh chứ không đặt biến môi trường của notebook, để chạy lại từng ô
theo thứ tự nào cũng đúng.

Muốn dùng cả hai T4 thì phải hạ `--grad-accum` xuống 4 để giữ batch hiệu dụng 16, và
ghi rõ điều đó trong báo cáo.

In [ ]:
# Kaggle chi cho ghi vao /kaggle/working. Gioi han output cua mot phien la 20 GB.
REPO = "https://github.com/ICY825/SummariseVietNamese.git"
DIR = "/kaggle/working/BTL_DL"

import os
if not os.path.isdir(DIR):
    !git clone -q {REPO} {DIR}

# `os.chdir` chu khong phai `%cd`: cac o `!` phia sau chay bang cwd cua kernel, va
# `os.chdir` doi dung cai do. `%cd {DIR}` phu thuoc vao viec magic co no chuoi bien
# hay khong, khac nhau giua cac ban IPython.
os.chdir(DIR)
!git pull -q
!git log --oneline -1
print("cwd:", os.getcwd())

Nếu repo để **private**, `git clone` sẽ hỏi mật khẩu rồi treo. Khi ấy vào
*Add-ons → Secrets*, lưu một GitHub token tên `GH_TOKEN`, rồi clone bằng:

```python
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GH_TOKEN")
!git clone -q https://{tok}@github.com/ICY825/SummariseVietNamese.git {DIR}
```

In [ ]:
# Tu kiem tra: khong can mang, khong can GPU, xong trong vai giay.
# Chay truoc de biet chac ROUGE va bootstrap van dung trong moi truong Kaggle.
!python src/eval/selftest.py | tail -3
!python src/models/selftest.py | tail -3

## Chạy thử đường ống trước — 3 phút

`--max-steps 5` dừng huấn luyện sau 5 bước, `--eval-limit 20` chỉ sinh 20 bài. Không
lấy số liệu từ lần chạy này (tên file tự mang hậu tố `_thu20` để khỏi lẫn), mục đích
là để lỗi phiên bản thư viện hay lỗi tải dữ liệu lộ ra **bây giờ**, chứ không phải ở
phút thứ 60 của lần chạy thật.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python src/models/vit5.py \
    --train-split train_2k --eval-split val \
    --max-steps 5 --eval-limit 20 \
    --out /kaggle/working/runs

## Chạy thật

`--out` phải trỏ vào `/kaggle/working`: đó là thư mục duy nhất được ghi và là thứ duy
nhất còn lại sau khi phiên kết thúc. Kết quả ra ba file — bảng chỉ số, bản dự đoán, và
`run.json` (hồ sơ lần chạy) — kèm một bản sao trong `--out` để sống sót khi ngắt phiên.

`fp16` tự bật vì T4 không hỗ trợ bf16. Nếu `eval_loss` thành `NaN` thì chạy lại với
`--fp16 off`, chậm hơn khoảng 33%.

Đổi `--train-split` thành `train_10k` rồi `train_20k` để dựng đường cong học của tuần 5.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python src/models/vit5.py \
    --model VietAI/vit5-base \
    --train-split train_5k --eval-split val \
    --epochs 3 --lr 3e-5 --batch 2 --grad-accum 8 \
    --out /kaggle/working/runs

In [ ]:
# Xem lai ket qua: bang chi so va ho so lan chay.
import json, pathlib

tables = sorted(pathlib.Path("results/tables").glob("*_run.json"))
for p in tables:
    r = json.loads(p.read_text(encoding="utf-8"))
    print("=" * 70)
    print(p.name)
    print("  GPU        ", r["env"]["gpu"], "| fp16", r["env"]["fp16"])
    print("  train      ", r["data"]["train_split"], r["data"]["n_train"], "bai")
    print("  lr/epochs  ", r["args"]["lr"], "/", r["args"]["epochs"])
    print("  sinh       ", r["generation"])
    if r["train"]:
        print("  phut       ", r["train"]["minutes"])
        print("  checkpoint ", r["train"]["best_checkpoint"], "eval_loss", r["train"]["best_eval_loss"])
        ev = [(h.get("epoch"), h["eval_loss"]) for h in r["train"]["log_history"] if "eval_loss" in h]
        print("  eval_loss  ", ev)
    print("  rouge1     ", round(r["scores"]["corpus"]["rouge1"]["mean"], 2))
    print("  do dai     ", round(r["scores"]["length"]["mean_syllables"], 1), "am tiet")

## Train ngầm — đóng trình duyệt vẫn chạy tiếp

Phiên tương tác chết khi mất mạng hoặc khi để yên quá lâu. Kaggle có sẵn cách chạy
tách rời, và đây là điểm hơn hẳn Colab free:

**Save Version → chọn `Save & Run All (Commit)` → Save.**

Kaggle chép notebook sang một máy khác rồi chạy **toàn bộ các ô từ đầu đến cuối**,
không cần trình duyệt mở. Tắt máy đi ngủ cũng được. Vài điều phải nhớ:

- Bản chạy ngầm bắt đầu từ môi trường sạch, nên ô `git clone` và ô tải dữ liệu đều
  chạy lại. Đừng để ô nào chờ nhập tay.
- Xong thì vào tab **Output** của version đó để tải `results/` và `runs/` về. Đây là
  nơi lấy `run.json` và file dự đoán.
- Giới hạn vẫn là **12 giờ** và output **20 GB**.
- Theo dõi tiến độ ở trang notebook, mục *Versions* — log in ra được xem trực tiếp.

Muốn chắc chắn không mất công, trước khi Commit hãy chạy tay ô "chạy thử đường ống"
một lần: một bản Commit hỏng ở phút thứ 50 vẫn tiêu tốn đúng ngần ấy giờ trong quota.

In [ ]:
# TUY CHON — chi chay khi da tai ket qua ve, hoac khi output qua nang.
# Moi `checkpoint-*` cua ViT5-base nang khoang 2,7 GB (trong so + trang thai optimizer);
# `save_total_limit=2` nen co the ton ~5,4 GB, du sat gioi han 20 GB cua output.
# `final/` la ban da duoc `load_best_model_at_end` chon, giu lai la du de tuan 5 khao
# sat tham so sinh bang `--no-train --model <duong dan final>`.
!du -sh /kaggle/working/runs/* 2>/dev/null
# !rm -rf /kaggle/working/runs/*/checkpoint-*

## Chạy tiếp khi bị cắt giữa chừng

`vit5.py` ghi checkpoint sau mỗi epoch, nên lần chạy sau nối tiếp được bằng `--resume`.
Trên Kaggle, checkpoint của phiên trước không tự có mặt: vào *Add-ons → Add data →
Your Work*, thêm output của version cũ làm input (nó nằm ở `/kaggle/input/...`), chép
sang `/kaggle/working/runs`, rồi chạy lại lệnh huấn luyện kèm `--resume`.

Với `train_5k` (60 phút) thì việc này hầu như không cần; nó dành cho `train_20k`.